In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.attention as attn

In [2]:
B = 5  ## Batch 
C = 16 ## number of Channel 
R = 4 ## dimension of signals (just assumed)
L = 1000 ## length of time series

X = torch.rand((B, C, R, L))

In [4]:
class CNNTokenizer(nn.Module):
    def __init__(self, Cin, token_dim =128, patch_stride=8, depth = 2 ):
        """
        Cin: C*R
        D: Dimension of tokens 
        depth: how many convolutional stages stacked 
        """
        super().__init__()
        layers = []
        ch = Cin
        for i in range(depth-1):
            layers += [
                nn.Conv1d(ch, token_dim, kernel_size=7, stride=patch_stride if i == 0 else 8, padding = 3, groups = 1),
                nn.BatchNorm1d(token_dim),
                nn.GELU()
            ]
            ch = token_dim
            
        self.encoder = nn.Sequential(*layers)
        self.proj = nn.Conv1d(token_dim, token_dim, kernel_size=1)
        self.pos = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, Cin, L)
        z = self.encoder(x)              # (B, D, T)
        z = self.proj(z)                 # (B, D, T) # T is number of tokens divided by patch stride, 1000/(8*2)
        tokens = z.transpose(1, 2)       # (B, T, D)
        
        # build (or resize) learned pos embeddings once T is known
        B, T, D = tokens.shape
        if (self.pos is None) or (self.pos.shape[1] != T):
            self.pos = nn.Parameter(torch.zeros(1, T, D))
            nn.init.trunc_normal_(self.pos, std=0.02)
        return tokens + self.pos 

In [7]:
X1 = X.view(B, C*R, L)
tok = CNNTokenizer(C*R, 128, 8, 2)
tokens = tok(X1)
tokens.shape
tokens.mean(dim=0).shape

torch.Size([125, 128])

In [3]:
class ActionClassifier(nn.Module):
    def __init__(self, n_channels: int, n_times: int,
                 n_behavior_classes: int, n_gesture_classes: int,
                 dropout_rate: float = 0.3, task: str = 'behavior', device = 'cuda'):
        super().__init__()
        self.task = task
        self.device = device
        self.trunk = nn.Sequential(
            nn.Conv1d(n_channels, n_channels * 2, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(n_channels * 2), nn.ReLU(), nn.MaxPool1d(3,2,1), nn.Dropout(dropout_rate),
            nn.Conv1d(n_channels * 2, n_channels * 2, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(n_channels * 2), nn.ReLU(), nn.MaxPool1d(3,2,1), nn.Dropout(dropout_rate),
            nn.Conv1d(n_channels * 2, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(3,2,1), nn.Dropout(dropout_rate),
        ).to(self.device)

        # compute pooled length roughly; we’ll adaptively pool anyway
        self.time_after_pool = max(1, n_times // 8)

        self.proj = nn.Linear(128, 128).to(self.device)  # token projection per time step
        # heads are fed a time-aggregated token (mean over time)
        self.behavior_head = nn.Linear(128, n_behavior_classes).to(self.device)
        self.gesture_head  = nn.Linear(128, n_gesture_classes).to(self.device)

    def forward(self, x, return_tokens: bool = False, pool_tokens_to: int | None = None):
        # x: (B, C, T) (batch size, channels, time steps)
        h = self.trunk(x)                     # (B, 128, T')
        if pool_tokens_to is not None:
            h = torch.nn.functional.adaptive_avg_pool1d(h, pool_tokens_to)
        else:
            if h.size(-1) != self.time_after_pool:
                h = torch.nn.functional.adaptive_avg_pool1d(h, self.time_after_pool)

        # tokens = projected per-timestep features
        tokens = self.proj(h.transpose(1,2))  # h.T(B, T', 128) @ (128, 128) -> (B, T', 128)

        # global representation for classification
        glob = tokens.mean(dim=1)             # (B, 128) Take mean over time dimension

        out = {}
        if self.task in ('behavior','both'):
            out['behavior_logits'] = self.behavior_head(glob)
        if self.task in ('gesture','both'):
            out['gesture_logits']  = self.gesture_head(glob)

        if return_tokens:
            out['tokens'] = tokens

        return out

    @torch.no_grad()
    def tokens(self, x):
        """Return per-window 'tokens' (B, T', 128) for downstream usage."""
        h = self.trunk(x)
        h = torch.nn.functional.adaptive_avg_pool1d(h, self.time_after_pool)
        return self.proj(h.transpose(1,2))

    @torch.no_grad()
    def tokens_from_raw(self, x, pool_tokens_to: int | None = None ):
        """
        return per-sequence tokens with optional pooling to fixed K
        Pooling over the time series length dimension
        """
        h = self.trunk(x) ## (B, 128, T')
        if pool_tokens_to is not None:
            h = torch.nn.functional.adaptive_avg_pool1d(h, pool_tokens_to) ## (B, 128, K)
        return self.proj(h.transpose(1, 2))  # (B, K, 128)